# EDA Explorer — パチスロホール分析フレームワーク

## クイックスタート
1. セル順に実行（Shift+Enter）
2. ホール名・次元・フィルタを変えて探索
3. Tier A パターンが出たら `export_tier_a_drafts()` で instinct ドラフトを生成

---

In [1]:
# ライブラリ読み込み（常にここから開始）
import sys
sys.path.insert(0, '..')

import pandas as pd
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:,.1f}'.format)

from eda.core import load_hall_df, scan_dimension, cross_hall_scan, lag_analysis, HALL_DBS
from eda.auto_explorer import full_scan, summarize_scan, FILTERS, ALL_DIMENSIONS
from eda.instinct_generator import export_tier_a_drafts, export_summary_drafts

print('利用可能なホール:', list(HALL_DBS.keys()))
print('利用可能なフィルタ:', list(FILTERS.keys()))
print('スキャン可能な次元:', ALL_DIMENSIONS)

利用可能なホール: ['みとや', '蒲田7', '蒲田1', '楽園', 'レイトギャップ', 'ARROW', 'ヒロキ', 'ザシティ', '金時']
利用可能なフィルタ: ['all', 'x_day_only', 'event_only', 'weekend_only', 'non_event']
スキャン可能な次元: ['day_of_week', 'date_digit', 'dd', 'dd_mod10', 'dd_group', 'machine_digit', 'machine_name', 'is_x_day', 'is_weekend', 'is_any_event', 'weekday_nth']


---
## 1. 単一次元スキャン（仮説検証型）

`scan_dimension(ホール名, [次元リスト], filters=フィルタ, min_n=最小N)`

**出力の読み方:**
- `tier`: A=実行レベル / B=参考 / C=観察中 / Avoid=回避
- `p_value`: < 0.10 で次元全体が有意
- `epsilon_sq`: 効果量（0.01=小, 0.06=中, 0.14=大）
- `ci_lo/ci_hi`: 95% Bootstrap 信頼区間
- `spearman_rho`: 月間トレンド（+なら上昇傾向, -なら低下傾向）

In [2]:
# 例1: みとや — DD系統 × 曜日
result = scan_dimension(
    hall_name  = 'みとや',
    group_cols = ['dd_group', 'day_of_week'],
    min_n      = 8,
    # filters  = {'is_x_day': 1},  # イベント日限定に絞る場合
)

display_cols = ['dd_group', 'day_of_week', 'n', 'avg_diff', 'median_diff',
                'plus_rate', 'tier', 'p_value', 'epsilon_sq', 'ci_lo', 'ci_hi', 'spearman_rho']
result[[c for c in display_cols if c in result.columns]]

,dd_group,day_of_week,n,avg_diff,median_diff,plus_rate,tier,p_value,epsilon_sq,ci_lo,ci_hi,spearman_rho
0,その他,土,265,277,-341,41.5,C,0.6,0.0,-113,628,NaN
1,7系,日,261,99,-326,39.8,C,0.6,0.0,-208,430,NaN
2,その他,金,249,97,-362,38.2,C,0.6,0.0,-132,333,NaN


In [3]:
# 例2: みとや — イベント日限定 × 台番号末尾
result2 = scan_dimension(
    hall_name  = 'みとや',
    group_cols = ['machine_digit'],
    filters    = {'is_x_day': 1},
    min_n      = 8,
)
result2[[c for c in display_cols if c in result2.columns]]

,n,avg_diff,median_diff,plus_rate,tier,p_value,epsilon_sq,ci_lo,ci_hi,spearman_rho
0,26,637,-416,42.3,C,0.9,0.0,-467,1885,NaN
1,26,406,-224,42.3,C,0.9,0.0,-464,1339,NaN
2,27,349,-29,44.4,C,0.9,0.0,-569,1353,NaN
3,26,213,-372,42.3,C,0.9,0.0,-1268,2207,NaN
4,28,86,-116,42.9,C,0.9,0.0,-694,895,NaN
5,26,33,-206,42.3,C,0.9,0.0,-711,796,NaN
6,26,7,-547,38.5,C,0.9,0.0,-914,916,NaN
7,25,-39,-553,36.0,C,0.9,0.0,-677,745,NaN
8,25,-287,-529,36.0,C,0.9,0.0,-917,452,NaN
9,26,-439,-782,30.8,C,0.9,0.0,-1513,561,NaN


In [4]:
# 例3: みとや — x_day内 DD × 日付末尾
result3 = scan_dimension(
    hall_name  = 'みとや',
    group_cols = ['dd', 'date_digit'],
    filters    = {'is_x_day': 1},
    min_n      = 5,
)
result3[[c for c in display_cols if c in result3.columns]]

,n,avg_diff,median_diff,plus_rate,tier,p_value,epsilon_sq,ci_lo,ci_hi,spearman_rho
0,261,99,-326,39.8,C,NaN,NaN,-208,430,NaN


---
## 2. 自動総当たりスキャン（Auto-Explorer）

`full_scan(ホール名, max_dims=次元数上限, filters=フィルタ, min_n=最小N)`

**注意:** max_dims=2 で全2次元組み合わせをスキャン。時間がかかる場合は `dimensions=` で列を絞る。

In [5]:
# 全次元 1次元スキャン（高速、まず全体把握に）
scan1d = full_scan(
    hall_name = 'みとや',
    max_dims  = 1,
    min_n     = 8,
)
summary1d = summarize_scan(scan1d, tier_filter=['A', 'B'])
summary1d[['dimension', 'n', 'avg_diff', 'median_diff', 'plus_rate', 'tier', 'p_value', 'epsilon_sq']]

[full_scan] みとや — データ読み込み中...
  [1/11] day_of_week ... no signal
  [2/11] date_digit ... no signal
  [3/11] dd ... TierA=0, TierB=3
  [4/11] dd_mod10 ... TierA=0, TierB=1
  [5/11] dd_group ... TierA=0, TierB=1
  [6/11] machine_digit ... no signal
  [7/11] machine_name ... TierA=0, TierB=6
  [8/11] is_x_day — SKIP (binary)
  [9/11] is_weekend — SKIP (binary)
  [10/11] is_any_event — SKIP (binary)
  [11/11] weekday_nth ... no signal

✓ スキャン完了: 4 / 11 次元にシグナルあり


,dimension,n,avg_diff,median_diff,plus_rate,tier,p_value,epsilon_sq
0,dd,4509,280,-229,45.6,B,0.0,0.0
1,dd_group,13509,242,-274,44.9,B,0.0,0.0
2,dd_mod10,13509,242,-274,44.9,B,0.0,0.0
3,dd,4499,234,-253,45.3,B,0.0,0.0
4,dd,4501,213,-332,43.7,B,0.0,0.0
5,machine_name,30,213,-142,46.7,B,0.0,0.0
6,machine_name,277,211,-238,43.7,B,0.0,0.0
7,machine_name,1097,180,-397,44.5,B,0.0,0.0
8,machine_name,124,156,-82,45.2,B,0.0,0.0
9,machine_name,565,140,-206,43.9,B,0.0,0.0


In [6]:
# 2次元スキャン（特定次元セットに絞った探索）
scan2d = full_scan(
    hall_name  = 'みとや',
    max_dims   = 2,
    min_n      = 8,
    dimensions = ['dd_group', 'day_of_week', 'date_digit', 'machine_digit', 'machine_name'],
)
summary2d = summarize_scan(scan2d, tier_filter=['A'])
summary2d[['dimension', 'n', 'avg_diff', 'plus_rate', 'tier', 'p_value', 'epsilon_sq']]

[full_scan] みとや — データ読み込み中...
  [1/15] dd_group ... TierA=0, TierB=1
  [2/15] day_of_week ... no signal
  [3/15] date_digit ... no signal
  [4/15] machine_digit ... no signal
  [5/15] machine_name ... TierA=0, TierB=6
  [6/15] dd_group × day_of_week ... no signal
  [7/15] dd_group × date_digit ... no signal
  [8/15] dd_group × machine_digit ... TierA=0, TierB=8
  [9/15] dd_group × machine_name ... TierA=12, TierB=50
  [10/15] day_of_week × date_digit ... no signal
  [11/15] day_of_week × machine_digit ... TierA=0, TierB=5
  [12/15] day_of_week × machine_name ... TierA=4, TierB=5
  [13/15] date_digit × machine_digit ... TierA=0, TierB=5
  [14/15] date_digit × machine_name ... TierA=4, TierB=5
  [15/15] machine_digit × machine_name ... TierA=6, TierB=70

✓ スキャン完了: 9 / 15 次元にシグナルあり


,dimension,n,avg_diff,plus_rate,tier,p_value,epsilon_sq
0,date_digit × machine_name,12,4464,83.3,A,0.0,0.0
1,day_of_week × machine_name,12,4464,83.3,A,0.0,0.0
2,day_of_week × machine_name,17,2540,64.7,A,0.0,0.0
3,date_digit × machine_name,17,2540,64.7,A,0.0,0.0
4,machine_digit × machine_name,16,1467,56.2,A,0.0,0.0
5,dd_group × machine_name,29,1158,58.6,A,0.0,0.0
6,machine_digit × machine_name,60,1081,56.7,A,0.0,0.0
7,dd_group × machine_name,16,1078,56.2,A,0.0,0.0
8,dd_group × machine_name,12,1053,58.3,A,0.0,0.0
9,dd_group × machine_name,535,1026,55.1,A,0.0,0.0


In [7]:
# イベント日限定で2次元スキャン
scan_event = full_scan(
    hall_name  = 'みとや',
    max_dims   = 2,
    min_n      = 5,
    filters    = FILTERS['x_day_only'],
    dimensions = ['dd', 'day_of_week', 'date_digit', 'machine_digit'],
)
summary_event = summarize_scan(scan_event, tier_filter=['A', 'B'])
summary_event[['dimension', 'n', 'avg_diff', 'plus_rate', 'tier', 'p_value', 'epsilon_sq']]

[full_scan] みとや — データ読み込み中...
  [1/10] dd ... no signal
  [2/10] day_of_week ... no signal
  [3/10] date_digit ... no signal
  [4/10] machine_digit ... no signal
  [5/10] dd × day_of_week ... no signal
  [6/10] dd × date_digit ... no signal
  [7/10] dd × machine_digit ... no signal
  [8/10] day_of_week × date_digit ... no signal
  [9/10] day_of_week × machine_digit ... no signal
  [10/10] date_digit × machine_digit ... no signal

✓ スキャン完了: 0 / 10 次元にシグナルあり
指定 Tier のパターンが見つかりませんでした。


KeyError: "None of [Index(['dimension', 'n', 'avg_diff', 'plus_rate', 'tier', 'p_value',\n       'epsilon_sq'],\n      dtype='str')] are in the [columns]"

---
## 3. クロスホール比較

同じ次元を複数ホールで比較。`universality='universal'` のパターンが共通ルール候補。

In [ ]:
# DD系統 × 曜日 を全ホールで比較
cross = cross_hall_scan(
    group_cols = ['dd_group', 'day_of_week'],
    min_n      = 8,
    # halls    = ['みとや', '蒲田7'],  # 特定ホールのみ
)

if not cross.empty:
    print('=== 共通パターン（2ホール以上でTier A/B）===')
    universal = cross[cross['universality'] == 'universal']
    display(universal[['hall','dd_group','day_of_week','n','avg_diff','plus_rate','tier','n_halls_tier_ab']])

    print('\n=== ホール固有 Tier A パターン ===')
    specific_a = cross[(cross['universality'] == 'hall_specific') & (cross['tier'] == 'A')]
    display(specific_a[['hall','dd_group','day_of_week','n','avg_diff','plus_rate','tier']])

---
## 4. ラグ分析（据え・上げ・リバウンド）

- `rebound_rate`: 前日マイナス→翌日プラスになる確率（高い = リバウンド傾向強）
- `streak_rate`: 前日プラス→翌日もプラスになる確率（高い = 据え継続傾向強）

In [ ]:
# みとや — 機種名単位のラグ分析
lag = lag_analysis(
    hall_name = 'みとや',
    group_col = 'machine_name',
    lag       = 1,
    min_n     = 5,
)
lag

In [ ]:
# x_day翌日効果の直接検証
df = load_hall_df('みとや')

daily = df.groupby('date').agg(
    avg_diff = ('diff', 'mean'),
    is_x_day = ('is_x_day', 'first'),
).reset_index().sort_values('date')

daily['prev_x_day'] = daily['is_x_day'].shift(1)
daily['prev_diff']  = daily['avg_diff'].shift(1)
daily = daily.dropna()

after_xday     = daily[daily['prev_x_day'] == 1]['avg_diff']
after_non_xday = daily[daily['prev_x_day'] == 0]['avg_diff']

print(f'x_day翌日  : avg={after_xday.mean():.0f}, plus_rate={(after_xday > 0).mean()*100:.1f}%, n={len(after_xday)}')
print(f'非x_day翌日: avg={after_non_xday.mean():.0f}, plus_rate={(after_non_xday > 0).mean()*100:.1f}%, n={len(after_non_xday)}')

---
## 5. Instinct ドラフト生成

Tier A パターンを `document/instincts/` に YAML ドラフトとして出力する。
**必ず人間がレビューして `【要記入】` 部分を埋めてから正式採用すること。**

In [ ]:
# scan_dimension の結果から Tier A ドラフトを生成
# export_tier_a_drafts(
#     scan_result = result,
#     group_cols  = ['dd_group', 'day_of_week'],
#     hall_name   = 'みとや',
# )

# full_scan の結果から全 Tier A ドラフトを一括生成
# export_summary_drafts(
#     summarize_result = summary2d,
#     hall_name        = 'みとや',
# )

---
## 6. カスタム分析

`load_hall_df()` でデータを取得し Pandas で自由に操作できる。

In [ ]:
# テンプレート: 機種名でフィルタしてDD×曜日分析
df = load_hall_df('みとや')

# 機種名でフィルタ
df_jug = df[df['machine_name'].str.contains('ジャグラー', na=False)]

result_jug = scan_dimension(
    hall_name  = 'みとや',
    group_cols = ['dd_group', 'day_of_week'],
    min_n      = 5,
    df         = df_jug,  # フィルタ済みDataFrameを渡す
)
result_jug[[c for c in display_cols if c in result_jug.columns]]